# 01 — Vescovo Dataset Schema

SiteLens AI — Layer-0 schema reference.

**Dataset:** Vescovo et al. (2025). Noto Peninsula 2024 earthquake building damage assessment.  
Zenodo doi:10.5281/zenodo.11055711. CC-BY 4.0. n = 140,208 buildings.

Run this notebook to inspect the GPKG schema, value distributions, and CRS.  
Output is reference material only — no model training happens here.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import geopandas as gpd
import pandas as pd

GPKG_PATH = "../data/raw/Noto_Peninsula_Damage_2_5.gpkg"
LAYER = "v2.5"

In [ ]:
# Schema: column names and types
gdf = gpd.read_file(GPKG_PATH, layer=LAYER, rows=5)
print("=== COLUMNS ===")
print(gdf.dtypes.to_string())
print("\n=== CRS ===")
print(gdf.crs)
print("\n=== SAMPLE ROWS ===")
print(gdf.drop(columns="geometry").head(3).to_string())

In [ ]:
# Value distributions across the full dataset
gdf_full = gpd.read_file(GPKG_PATH, layer=LAYER)
print(f"Total records: {len(gdf_full):,}")
print("\ndamage_val distribution:")
print(gdf_full["damage_val"].value_counts())
print("\nconf distribution:")
print(gdf_full["conf"].value_counts())
print("\nHazard flags (sum across all destroyed buildings):")
destroyed = gdf_full[gdf_full["damage_val"] == 1]
print(f"  GSI_fire:          {destroyed['GSI_fire'].sum():,}")
print(f"  GSI_tsunami:       {destroyed['GSI_tsunami'].sum():,}")
print(f"  GSI_slope_failure: {destroyed['GSI_slope_failure'].sum():,}")
print(f"  seismic only:      {((destroyed['GSI_fire']==0)&(destroyed['GSI_tsunami']==0)&(destroyed['GSI_slope_failure']==0)).sum():,}")

In [ ]:
# Hero zone: Wajima Asaichi fire district
from src.data.load_vescovo import load_fire_zone
fire_gdf = load_fire_zone()
print(f"Asaichi fire zone — destroyed+fire buildings: {len(fire_gdf)}")
print(fire_gdf[["damage_val", "GSI_fire", "GSI_tsunami", "USGS_MMI", "municipality"]].head())